In [100]:
from statsmodels.stats.multitest import multipletests
from scipy.stats import mannwhitneyu, kruskal
from collections import defaultdict
from datetime import datetime
import pandas as pd
import numpy as np
import re
import json
import os


In [101]:
data_dir = "./data"
processed_dir = os.path.join(data_dir, "processed")
session_dir = os.path.join(processed_dir, "merged")

gameplay_dir = os.path.join(session_dir, "Gameplay")
traces_path = os.path.join(gameplay_dir, "traces.ndjson")


In [102]:
with open(traces_path, "r", encoding="utf-8") as f:
	print(repr(f.read()[:200]))
	

'{"actor": {"account": {"name": "684837aae48b5a00221a37c9_fwme", "homePage": "https://simva-beta.e-ucm.es"}}, "result": {"extensions": {"https://w3id.org/xapi/seriousgame/extensions/Sexuality": "hetero'


In [103]:
traces = []
invalid_count = 0

with open(traces_path, "r", encoding="utf-8") as f:
	for line in f:
		line = line.strip()
		if line:
			try:
				traces.append(json.loads(line))
			except json.JSONDecodeError:
				invalid_count += 1

print("Total valid traces:", len(traces))
print("Total invalid traces:", invalid_count)


Total valid traces: 98930
Total invalid traces: 0


In [104]:
NODE_EXT_ID = "https://w3id.org/xapi/seriousgame/extensions/Node"
RESPONSE_EXT_ID = "https://w3id.org/xapi/seriousgame/extensions/Response"
CHOICE_OBJ_ID = "https://w3id.org/xapi/seriousgames/activity-types/dialog-tree/OptionSelect"


In [105]:
target_nodes = {
	"Scene6Bedroom.phone.choices2": "send_more_nudes",
	"Scene6BedroomRouteA1.phone.choices2": "agree_to_meet_harasser",
	"Scene6LunchRouteB.interruption.choices": "confess_to_parents"
}

END_NODES = {
	"END_MEET": "meet_harasser_in_person",
	"END_TELL": "confess_truth",
	"END_LIE": "hide_truth"
}


def parse_time(timestamp):
	if not timestamp:
		return datetime.fromisoformat("1970-01-01T00:00:00+00:00")

	return datetime.fromisoformat(timestamp.replace("Z", "+00:00"))


def classify_path(path):
	options = [option for _, option in path]

	if "Enviar otra foto." in options and "Aceptar." in options:
		return "END_MEET"

	if "Decir la verdad." in options:
		return "END_TELL"

	if "Mentir." in options:
		return "END_LIE"

	return None


def extract_sequence(traces):
	traces = sorted(
		traces,
		key=lambda trace: parse_time(trace.get("timestamp"))
	)

	sequence = []

	for trace in traces:
		if trace.get("object", {}).get("id") != CHOICE_OBJ_ID:
			continue

		extensions = trace.get("result", {}).get("extensions", {}) or {}

		node = extensions.get(NODE_EXT_ID)
		option = extensions.get(RESPONSE_EXT_ID)

		if node in target_nodes and option:
			sequence.append((node, option))

	return sequence


# Agrupar trazas por usuario
user_traces = defaultdict(list)

for trace in traces:
	user = trace.get("actor", {}).get("account", {}).get("name")

	if user:
		user_traces[user].append(trace)

# Usuarios de cada final
users_by_end = defaultdict(list)

for user, user_trace_list in user_traces.items():
	path = extract_sequence(user_trace_list)
	end = classify_path(path)

	if end:
		users_by_end[end].append(user)


# Mostrar resultados
for end, users in users_by_end.items(): 
	print(f"{END_NODES[end]}: {len(users)} usuarios")
		

meet_harasser_in_person: 33 usuarios
hide_truth: 9 usuarios
confess_truth: 62 usuarios


In [106]:
def load_json(path):
	with open(path, encoding="utf-8") as f:
		return json.load(f)

def load_tests(base_dir: str):
	data = defaultdict(lambda: {
		"pre": {"code": None, "full": None},
		"post": {"code": None, "full": None},
	})

	for phase in ["Pre", "Post"]:
		phase_key = phase.lower()
		phase_dir = os.path.join(base_dir, phase)

		for file in os.listdir(phase_dir):
			if file.endswith(".json"):
				file_path = os.path.join(phase_dir, file)
				content = load_json(file_path)

				kind = "code" if "code" in file else "full"

				for user_id, user_data in content.items():
					data[user_id][phase_key][kind] = user_data

	return dict(data)


In [107]:
def parse_likert(v: str):
	m = re.match(r"AO0?(\d)", v)
	if m:
		return int(m.group(1))

	return None

def extract_data(code: dict, full: dict):
	out = {}
	items = {}

	# pd.to_numeric -> convertir el valor a un número
	# errors="coerce" -> si no puede convertir el valor a número, devuelve NaN
	age = pd.to_numeric(full.get("Edad"), errors="coerce")

	full_keys = list(full.keys())

	for idx, (k, v) in enumerate(code.items()):
		if k.startswith("G02Q03[SQ"):		
			m = re.search(r"\[SQ(\d+)\]", k)
			if m:
				q_id = f"Q{m.group(1)}"
				out[q_id] = parse_likert(v)
				items[q_id] = full_keys[idx]

	return items, out, age

In [108]:
tests = load_tests(session_dir)


In [109]:
rows = []

for user_id, test in tests.items():
	pre_code = test["pre"]["code"]
	pre_full = test["pre"]["full"]
	post_code = test["post"]["code"]
	post_full = test["post"]["full"]

	if not all([pre_code, pre_full, post_code, post_full]):
		print("Skipping:", user_id)
		continue

	pre_items, pre_vals, age = extract_data(pre_code, pre_full)
	post_items, post_vals, _ = extract_data(post_code, post_full)

	common_ids = set(pre_vals) & set(post_vals)

	for id in common_ids:
		rows.append({
			"user_id": user_id,
			"age": age,
			"id": id,
			"item": pre_items[id],
			"pre": pre_vals[id],
			"post": post_vals[id]
		})

global_df = pd.DataFrame(rows)
global_df["diff"] = global_df["post"] - global_df["pre"]

print("Total users:", global_df["user_id"].nunique())
print("Total rows:", len(global_df))

display(global_df.head())


Total users: 104
Total rows: 2080


,user_id,age,id,item,pre,post,diff
0,684837aae48b5a00221a37c9_uzdq,13.0,Q19,¿Cómo de peligrosas consideras las siguientes ...,3,5,2
1,684837aae48b5a00221a37c9_uzdq,13.0,Q03,¿Cómo de peligrosas consideras las siguientes ...,5,4,-1
2,684837aae48b5a00221a37c9_uzdq,13.0,Q06,¿Cómo de peligrosas consideras las siguientes ...,2,4,2
3,684837aae48b5a00221a37c9_uzdq,13.0,Q04,¿Cómo de peligrosas consideras las siguientes ...,2,5,3
4,684837aae48b5a00221a37c9_uzdq,13.0,Q15,¿Cómo de peligrosas consideras las siguientes ...,3,4,1


In [114]:
for _, row in global_df.iterrows():
    print(row["id"], row["item"])
    

Q19 ¿Cómo de peligrosas consideras las siguientes acciones? [Publicar o tener visible en algún sitio la calle en la que vives en un perfil público de una red social]
Q03 ¿Cómo de peligrosas consideras las siguientes acciones? [Unirte a grupos o comunidades o servidores públicos]
Q06 ¿Cómo de peligrosas consideras las siguientes acciones? [Entablar amistad con alguien que no conoces en la vida real]
Q04 ¿Cómo de peligrosas consideras las siguientes acciones? [Hablar por chat (escrito o de voz) con alguien que no conoces en la vida real]
Q15 ¿Cómo de peligrosas consideras las siguientes acciones? [Publicar o tener visible en algún sitio tu cumpleaños o edad o fecha de nacimiento en un perfil público de una red social]
Q01 ¿Cómo de peligrosas consideras las siguientes acciones? [Tener tu perfil en público]
Q13 ¿Cómo de peligrosas consideras las siguientes acciones? [Publicar o tener visible en algún sitio tu número de teléfono en un perfil público de una red social]
Q11 ¿Cómo de peligrosa

In [ ]:
user_to_group = {}

for user in users_by_end["END_MEET"]:
	user_to_group[user] = "MEET"

for user in users_by_end["END_TELL"]:
	user_to_group[user] = "TELL"

for user in users_by_end["END_LIE"]:
	user_to_group[user] = "LIE"

df = global_df.copy()

df["group"] = df["user_id"].map(user_to_group)

df["change"] = df["post"] - df["pre"]


# Kruskal-Wallis para cada ítem
results = []
for item_id, df_item in df.groupby("id"):

	groups = {
		group: data["change"].to_numpy()
		for group, data in df_item.groupby("group")
	}

	# Necesitamos los 3 grupos
	if not all(group in groups for group in ["MEET", "TELL", "LIE"]):
		continue

	# Al menos 2 participantes por grupo
	if any(len(values) < 2 for values in groups.values()):
		continue

	# Test de Kruskal-Wallis
	# Se utiliza para comparar una variable entre tres o más grupos independientemente de la normalidad.
	# En este análisis se utiliza para comprobar si el cambio pre-post difiere entre los tres finales: MEET, TELL y LIE.
	# Hipótesis nula (H0): la distribución del cambio pre-post es la misma en los tres grupos.
	# Hipótesis alternativa (H1): al menos uno de los tres grupos presenta una distribución del cambio pre-post diferente.
	# Si p < 0.05, se rechaza H0 y existe evidencia de que al menos uno de los grupos difiere de los demás.
	# El test de Kruskal-Wallis únicamente indica si existe una diferencia global entre los tres grupos, 
	# pero no identifica qué grupos difieren entre sí. 
	# Por ello, cuando el resultado es significativo, se realizan comparaciones por pares mediante el test de Mann-Whitney U.
	stat, p = kruskal(
		groups["MEET"],
		groups["TELL"],
		groups["LIE"]
	)

	results.append({
		"id": item_id,
		
		"n_MEET": len(groups["MEET"]),
		"n_TELL": len(groups["TELL"]),
		"n_LIE": len(groups["LIE"]),

		"mean_change_MEET": np.mean(groups["MEET"]),
		"mean_change_TELL": np.mean(groups["TELL"]),
		"mean_change_LIE": np.mean(groups["LIE"]),

		"median_change_MEET": np.median(groups["MEET"]),
		"median_change_TELL": np.median(groups["TELL"]),
		"median_change_LIE": np.median(groups["LIE"]),

		"stat": stat,
		"p": p
	})


comparison_df = pd.DataFrame(results)

# https://cienciadedatos.net/documentos/19b_comparaciones_multiples_correccion_p-value_fdr
# FDR (Benjamini-Hochberg) para los 20 tests de Kruskal-Wallis.
# Se realiza un test de Kruskal-Wallis independiente para cada ítem Likert.
# Al realizar múltiples tests, aumenta la probabilidad de obtener falsos positivos por el azar.
# En cada test, si se fija alpha = 0.05, existe un 5% de probabilidad
# de rechazar la hipótesis nula cuando es verdadera.
# Al realizar 20 tests, esta probabilidad acumulada de obtener falsos positivos aumenta.
# La corrección de Benjamini-Hochberg ajusta la False Discovery Rate (FDR)
comparison_df["p_adj"] = multipletests(comparison_df["p"], method="fdr_bh")[1]

comparison_df["significant"] = comparison_df["p_adj"] < 0.05

# Seleccionar ítems para los post-hoc
# Se hacen post-hoc solamente cuando el Kruskal-Wallis tiene p_adj < 0.05
significant_items = comparison_df.loc[
	comparison_df["p_adj"] < 0.05,
	"id"
].tolist()

# Post-hoc por pares
pairwise_results = []

pairs = [
	("MEET", "TELL"),
	("MEET", "LIE"),
	("TELL", "LIE")
]

for item_id in significant_items:
	df_item = df[df["id"] == item_id]
	
	groups = {
		group: data["change"].to_numpy()
		for group, data in df_item.groupby("group")
	}

	for group1, group2 in pairs:
		x = groups[group1]
		y = groups[group2]

		if len(x) < 2 or len(y) < 2:
			continue

		# Test de Mann-Whitney U
		# Se utiliza para comparar una variable no parametríca entre dos grupos independientes.
		# Hipótesis nula (H0): la distribución del cambio pre-post es la misma en los dos grupos comparados.
		# Hipótesis alternativa (H1): la distribución del cambio pre-post es diferente entre los dos grupos.
		stat, p = mannwhitneyu(
			x,
			y,
			alternative="two-sided"
		)

		pairwise_results.append({
			"id": item_id,
			"group1": group1,
			"group2": group2,
			"n1": len(x),
			"n2": len(y),
			"median1": np.median(x),
			"median2": np.median(y),
			"stat": stat,
			"p": p
		})


pairwise_df = pd.DataFrame(pairwise_results)

# Cada pregunta constituye una familia de hipótesis independiente.
# Por este motivo, la corrección FDR se aplica por separado dentro
# de cada pregunta, considerando los 3 conjuntos:
# - MEET vs TELL
# - MEET vs LIE
# - TELL vs LIE
if not pairwise_df.empty:
    pairwise_df["p_adj"] = np.nan

    for item_id, indices in pairwise_df.groupby("id").groups.items():
        p_values = pairwise_df.loc[indices, "p"]

		# FDR de los post-hoc
        pairwise_df.loc[indices, "p_adj"] = multipletests(p_values, method="fdr_bh")[1]

    pairwise_df["significant"] = pairwise_df["p_adj"] < 0.05

# Resultados
print("COMPARACIÓN GLOBAL: KRUSKAL-WALLIS")
display(comparison_df)

print("POST-HOC MANN-WHITNEY")
display(pairwise_df)


COMPARACIÓN GLOBAL: KRUSKAL-WALLIS


,id,n_MEET,n_TELL,n_LIE,mean_change_MEET,mean_change_TELL,mean_change_LIE,median_change_MEET,median_change_TELL,median_change_LIE,stat,p,p_adj,significant
0,Q01,33,62,9,0.424242,0.693548,1.000000,0.0,1.0,1.0,2.379730,0.304262,0.547303,False
1,Q02,33,62,9,0.515152,0.338710,0.555556,0.0,0.0,0.0,2.227156,0.328382,0.547303,False
2,Q03,33,62,9,0.666667,0.758065,1.333333,1.0,1.0,1.0,1.119264,0.571419,0.761892,False
3,Q04,33,62,9,0.393939,0.548387,-0.111111,0.0,0.0,0.0,2.834498,0.242380,0.547303,False
4,Q05,33,62,9,0.212121,0.177419,0.111111,0.0,0.0,0.0,0.265138,0.875842,0.879191,False
5,Q06,33,62,9,0.696970,0.693548,0.222222,1.0,1.0,0.0,2.241939,0.325964,0.547303,False
6,Q07,33,62,9,0.515152,0.096774,0.222222,0.0,0.0,0.0,3.064238,0.216077,0.547303,False
7,Q08,33,62,9,0.454545,0.225806,0.222222,0.0,0.0,0.0,2.347117,0.309264,0.547303,False
8,Q09,33,62,9,0.606061,0.725806,0.888889,0.0,1.0,0.0,1.237194,0.538700,0.761892,False
9,Q10,33,62,9,0.757576,0.741935,1.111111,1.0,1.0,1.0,0.400856,0.818381,0.879191,False


POST-HOC MANN-WHITNEY


""
